In [73]:
from IPython.display import display, Math
import logging
logging.getLogger('pyEPR').setLevel(logging.ERROR)
%load_ext autoreload
%autoreload 2
%config IPCompleter.greedy = True
import numpy as np
import pyEPR as epr
import pandas as pd
#epr.__file__


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [37]:
from pathlib import Path
path_to_project='D:\Krishna\Krishna_wrk\HFSS_Projects'
pinfo = epr.ProjectInfo(project_path = path_to_project,project_name = 'QubitDesign',design_name='Hello1')

In [38]:
pinfo.junctions['j1']={'Lj_variable' : 'Lj',
                       'rect':'Rectangle3',
                       'line': 'JJ_Line',
                       'length': epr.parse_units('100um')}
pinfo.validate_junction_info()


In [39]:
# Sends the command to HFSS to run the analysis for the first setup. 
pinfo.setup.analyze()

0

In [40]:
# This is the core object for interacting with HFSS and running analysis within HFSS
eprd = epr.DistributedAnalysis(pinfo) #epr HFSS analysis

Design "Hello1" info:
	# eigenmodes    2
	# variations    1


In [41]:
%%capture
# This runs the EPR math for all modes and variations. 
# It calculates the participation ratios (p_mj) for your junctions.
# Try running with these flags to skip the part of the code that 
# crashes while trying to talk to the HFSS GUI
eprd.do_EPR_analysis();


## Energy participation in dielectric in various modes 

In [42]:
def get_energy_distribution(n):
    """
    Calculates and returns a beautiful Energy Distribution DataFrame for Mode n.
    """
    # 1. Setup: Switch to the requested Mode 'n'
    eprd.set_variation('0')
    eprd.set_mode(n) 

    # 2. Efficient Calculation
    # Step A: Calculate Substrate (and capture E_total for reuse)
    # Returns: participation, (Energy_in_obj, Total_Energy_System)
    p_sub, (E_sub, E_total) = eprd.calc_p_electric_volume('Sapphire', variation='0')

    # Step B: Calculate Vacuum (reuse E_total to skip re-integration)
    p_vac, (E_vac, _) = eprd.calc_p_electric_volume('CAVITY', variation='0', E_total=E_total)

    # 3. Build Data Dictionary (Dynamic Key Name!)
    # We use f"Mode {n}" so the header changes automatically (Mode 0, Mode 1, etc.)
    mode_header = f"Mode {n}"
    
    data = {
        mode_header: ["Sapphire", "Vacuum Cavity", "SUM (Check)"],
        "Energy (J)": [E_sub, E_vac, (E_sub + E_vac)],
        "Participation": [p_sub, p_vac, (p_sub + p_vac)],
        "Distribution (%)": [p_sub * 100, p_vac * 100, (p_sub + p_vac) * 100]
    }

    # 4. Create DataFrame
    df = pd.DataFrame(data)

    # 5. Apply "Beautiful" Formatting
    format_mapping = {
        "Energy (J)": "{:.4e}",
        "Participation": "{:.6f}",
        "Distribution (%)": "{:.2f}%"
    }
    
    print(f"--- Analyzing Energy Breakdown for Mode {n} ---")
    
    # Return the styled object so Jupyter displays it beautifully
    return (
        df.style
        .format(format_mapping)
        .hide(axis="index") # Hides the index numbers
        .set_properties(**{'text-align': 'left'}) # Optional: Left align text
        .set_table_styles([
            dict(selector='th', props=[('text-align', 'left')]) # Align headers left
        ])
    )


In [43]:
# --- EXAMPLE USAGE ---
# Just call the function with the mode number you want!
# For Mode 0 (Qubit)
get_energy_distribution(0)

--- Analyzing Energy Breakdown for Mode 0 ---


Mode 0,Energy (J),Participation,Distribution (%)
Sapphire,6.8908e-23,0.874215,87.42%
Vacuum Cavity,9.9147e-24,0.125785,12.58%
SUM (Check),7.8823e-23,1.000000,100.00%


In [44]:
# For Mode 1 (Resonator)
get_energy_distribution(1)

--- Analyzing Energy Breakdown for Mode 1 ---


Mode 1,Energy (J),Participation,Distribution (%)
Sapphire,1.1824e-21,0.041001,4.10%
Vacuum Cavity,2.7657e-20,0.958999,95.90%
SUM (Check),2.8839e-20,1.000000,100.00%


In [45]:
# Initialize (point it to the file created by DistributedAnalysis)
eprq = epr.QuantumAnalysis(eprd.data_filename)

	 Differences in variations:




## JJ's Parameters

In [ ]:
def get_junction_report(eprq, eprd, variation='0', junction_name='JJ_Line'):
    """
    Extracts Josephson Junction parameters and geometry.
    Returns a clean DataFrame without crashing on tuple formatting.
    """
    # 1. Constants
    h    = 6.62607015e-34
    e    = 1.60217663e-19
    Phi0 = h / (2 * e)

    # 2. Extract Data
    try:
        Ejs = eprq.get_Ejs(variation=variation)
        Ecs = eprq.get_Ecs(variation=variation)
        L_J_raw, C_J_raw = eprd.get_junctions_L_and_C(variation=variation)
    except:
        print("⚠️ Error retrieving Energy/L/C data. Check pyEPR variation.")
        return None

    # 3. Extract Geometry (Safe Method)
    try:
        length_m, unit_vec = eprd.get_junc_len_dir(variation, junction_name)
        length_um = length_m * 1e6
        # Format direction as a string immediately to avoid styling errors later
        direction_str = str(tuple(np.round(unit_vec, 2)))
    except:
        length_um = 0.0
        direction_str = "Unknown"

    # 4. Create Master DataFrame
    df_params = pd.DataFrame({
        'Ej (GHz)': Ejs,
        'Ec (GHz)': Ecs,
        'Lj (nH)': L_J_raw * 1e9,
        'Cj (fF)': C_J_raw * 1e15,
        'Ic (nA)': (2 * np.pi * (Ejs * 1e9 * h) / Phi0) * 1e9,
        'Length (µm)': length_um
    })

    # Add Direction separately
    df_params['Direction'] = direction_str

    print(f"--- Final Josephson Junction Report (Variation {variation}) ---")
    
    # 5. Apply Styling ONLY to numeric columns
    # We select all columns EXCEPT 'Direction' for the number formatting
    numeric_cols = df_params.select_dtypes(include=[np.number]).columns
    
    return df_params.style.format("{:,.4f}", subset=numeric_cols)

# --- CALL IT ---
display(get_junction_report(eprq, eprd))

--- Final Josephson Junction Report (Variation 0) ---


,Ej (GHz),Ec (GHz),Lj (nH),Cj (fF),Ic (nA),Length (µm),Direction
j1,11.6758,9.6851,14.0000,2.0000,23.5076,50.0000,"(0.0, 1.0, 0.0)"


## -- ENERGY ANALYSIS --

In [77]:
%%capture
# 1. THE CORE COMMAND: Analyzes everything
# This calculates f_1, chi, and participation for all variations.
eprq.analyze_all_variations(cos_trunc=8, fock_trunc=7) 

In [78]:
def get_classical_energies(eprq, variation='0'):
    """
    Extracts and cleans the Classical Energy breakdown from Ansys.
    Returns a beautiful DataFrame including U_norm.
    """
    # 1. Set Display Format
    pd.options.display.float_format = '{:.4e}'.format

    # 2. Get Raw Data
    df_raw = eprq.get_ansys_energies()

    # 3. Helper to extract numbers from dictionary cells
    def extract_val(cell):
        if isinstance(cell, dict):
            return list(cell.values())[0] # Extract value from {'j1': ...}
        return cell

    # 4. Build the Clean DataFrame
    df_clean = pd.DataFrame()
    
    # --- The Components ---
    df_clean['U_J_Inductive']  = df_raw['U_J_inds'].apply(extract_val)
    df_clean['U_J_Capacitive'] = df_raw['U_J_caps'].apply(extract_val)
    df_clean['U_H (Magnetic)'] = df_raw['U_H']
    df_clean['U_E (Electric)'] = df_raw['U_E']
    
    # --- The Totals ---
    df_clean['Total Inductive']  = df_raw['U_tot_ind']
    df_clean['Total Capacitive'] = df_raw['U_tot_cap']
    
    # --- The Normalization & Error ---
    df_clean['U_norm (Total Energy)'] = df_raw['U_norm']
    df_clean['Energy Diff (%)']  = df_raw['U_diff']

    # 5. Filter for variation
    if 'variation' in df_raw.index.names and variation:
        try:
            df_clean = df_clean.xs(variation, level='variation')
        except:
            pass 

    print("--- Classical Energies (The Physics Balance Sheet) ---")
    return df_clean.style.format("{:.4e}")

# CALL IT
display(get_classical_energies(eprq))

--- Classical Energies (The Physics Balance Sheet) ---


,U_J_Inductive,U_J_Capacitive,U_H (Magnetic),U_E (Electric),Total Inductive,Total Capacitive,U_norm (Total Energy),Energy Diff (%)
mode,,,,,,,,
0,3.9056e-23,7.5361e-25,3.3734e-25,3.9411e-23,3.9393e-23,4.0165e-23,4.0165e-23,9.7024e-03
1,1.3339e-23,7.7324e-25,1.4403e-20,1.4420e-20,1.4417e-20,1.4420e-20,1.4420e-20,1.3418e-04


## --- QUANTUM PARAMTER EXTRACTION ---

In [ ]:
def get_mode_analysis_report(eprq, eprd, variation='0'):
    """
    Extracts Quantum Parameters (Freq, Q, Participation, ZPF) for all modes.
    Returns a clean DataFrame.
    """
    # 1. Configuration
    pd.set_option('display.max_columns', None)
    pd.options.display.float_format = '{:,.4e}'.format
    
    # 2. Extract Raw Data
    # PM = Inductive Participation Matrix
    # SIGN = Sign of the ZPF
    # Phi_ZPF = Zero Point Fluctuations (flux)
    # PJ_cap = Capacitive Participation
    try:
        PM, SIGN, Om, EJ, Phi_ZPF, PJ_cap, PM_norm = eprq.get_epr_base_matrices(variation=variation)
        ansys_freqs = eprd.get_freqs_bare_pd(variation=variation)
    except:
        print("⚠️ Error: Could not retrieve EPR matrices. Run 'analyze_all_variations' first.")
        return None

    # 3. Build Table Data
    # Initialize with the common mode properties
    data_modes = {
        'Freq (GHz)': ansys_freqs['Freq. (GHz)'].values,
        'Q Factor': ansys_freqs['Quality Factor'].values,
    }
    
    mode_labels = [f"Mode {m}" for m in range(PM.shape[0])]
    num_junctions = PM.shape[1]

    # 4. Loop through each junction to add specific columns
    for j in range(num_junctions):
        j_name = f"j{j+1}" # e.g., j1, j2
        
        # Inductive Participation (The most important one)
        data_modes[f' Inductive Part'] = PM[:, j]
        
        # Zero Point Fluctuations (Flux)
        data_modes[f' Phi ZPF (φ0)'] = Phi_ZPF[:, j]
        
        # Capacitive Participation (Usually small)
        if j < PJ_cap.shape[1]: 
            data_modes[f' Capacitive Part'] = PJ_cap[:, j]
            
        # Sign Matrix (Direction of field)
        data_modes[f' Sign'] = SIGN[:, j]

    # 5. Create DataFrame
    df_modes = pd.DataFrame(data_modes, index=mode_labels)

    # 6. Display Setup
    print("\n" + "="*60)
    print(f" TABLE 1: MODE ANALYSIS (Variation {variation})")
    print("="*60)
    print(" • Freq (GHz): Eigenmode Frequency")
    print(" • Inductive Part: Energy stored in this Junction vs Total Energy")
    print(" • Phi ZPF (φ0): Quantum Zero-Point Flux fluctuations")
    
    # Return Styled DataFrame (Highlighting high participation modes)
    # We highlight Inductive Participation in Green because it identifies the Qubit
    return df_modes.style.format("{:,.4e}")

# --- HOW TO USE ---
display(get_mode_analysis_report(eprq, eprd))

# --- SELF-KERR(ANHARMONICITY) & CROSS-KERR  ---

In [ ]:
def get_chi_matrix(variation=0, numeric=True, real_only=True, display_result=True):
    
    # 1. Get chi matrix
    df_chis = eprq.get_chis(numeric=numeric)

    # 2. Safely determine variation key
    available_vars = df_chis.index.get_level_values('variation').unique()
    var_to_use = variation if variation in available_vars else str(variation)

    # 3. Extract variation
    df_chis_var = df_chis.xs(var_to_use, level='variation')

    # 4. Convert to real part
    if real_only:
        df_chis_var = df_chis_var.apply(np.real)

    # 5. Rename rows and columns
    mode_labels = [f"Mode {i}" for i in range(len(df_chis_var))]
    df_chis_var.index = mode_labels
    df_chis_var.columns = mode_labels

    # 6. Put unit in top-left corner correctly
    df_chis_var.columns.name = "χ (MHz)"

    # 7. Format display
    pd.options.display.float_format = '{:.3f}'.format

    return df_chis_var

get_chi_matrix()

χ (MHz),Mode 0,Mode 1
Mode 0,202.800,0.555
Mode 1,0.555,0.000


In [18]:
pinfo.disconnect()